# Phase 2 driver

A **driver only** (CLAUDE2.md "Code layout"). Every cell here imports `src/`
code to launch a run or look at a result. No loader, training loop, config
system or baseline logic is defined in this notebook — that lives in `src/`
with tests and is run through `python -m scripts.t20x_*`.

Run `jupyter lab` from the repo root. Strip outputs before committing:
`nbstripout notebooks/phase2.ipynb`.

In [ ]:
import os
import pathlib
import sys

# Work from the repo root so `import src...` and the anchored corpus paths resolve
# whether the kernel started here or in notebooks/.
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

In [ ]:
# T-201 — environment guard. Same check every entry point runs.
from src.env_check import check_env

check_env()

## T-202 — data loading

`src/data.py`. Native splits only for training (`hard rule 1`); `load_split`
asserts the origin so a machine-translated split can't slip in.

In [ ]:
from src import data as D

TASK = 2  # 2 = sustainability (binary), 3 = ESG themes (10-class)
frame = D.load_native(TASK, "hin")
print(len(frame), "rows", "| labels:", D.label_names(TASK))
frame["label"].value_counts(normalize=True)

In [ ]:
# Seeded, class-stratified train/dev/test. Same seed -> identical partition.
parts = D.stratified_split(frame, seed=0, dev=0.1, test=0.1)
{name: len(part) for name, part in parts.items()}

In [ ]:
# Tokenise a split and look at one padded batch. mBERT here because it needs no
# gated-repo access; swap for 'indicbert-v2' once its terms are accepted.
ds = D.load_dataset(TASK, "hin", "mbert-base", max_len=128)
batch = ds.collate([ds[i] for i in range(8)])
{k: (v.shape if hasattr(v, "shape") else v) for k, v in batch.items()}

## T-203 — training loop

Built in `src/train.py`; run the correctness gate with
`python -m scripts.t203_smoke --device cuda`. To watch a curve here, call
`src.config.run_training(cfg, device="cuda")` on a small `RunConfig` and plot
`result["history"]`.

## T-205 — VRAM budgets

Measured: `reports/vram_budget.md`. Re-probe with
`python -m scripts.t205_vram --full-epoch`. Both shipped configs train an epoch
under ~1.2 GiB, well inside the 3050.

## T-206 — fine-tune task 2 and task 3 separately

Two independent runs, each fine-tuning IndicBERT-v2 on its own native Hindi
split from a shipped config (`configs/train/task{2,3}_hin_indicbert.yaml`).
This cell *drives* `src/config.run_training` — the loop, checkpointing and
determinism all live in `src/`, not here (CLAUDE2.md "Code layout").

Change `SEED` and re-run a cell to fine-tune another seed; the full 3-seed
sweep with resumability and report-writing is `scripts/t206_baseline.py`.

In [ ]:
from pathlib import Path

from src.checkpoints import checkpoint_dir, checkpoint_path
from src.config import load_run_config, run_training
from src.experiments import log_run

SEED = 0  # change and re-run to fine-tune another seed
CONFIG = "configs/train/task2_hin_indicbert.yaml"

run = load_run_config(CONFIG, seed=SEED)
run_id = f"{Path(CONFIG).stem}_seed{SEED}"

metrics = run_training(run, device="cuda", work_dir=checkpoint_dir(run_id), progress=print)
log_run(run, run_id, metrics)

print(f"\nrun_id: {run_id}")
print(f"macro-F1 (test): {metrics['macro_f1']:.4f}  accuracy: {metrics['accuracy']:.4f}")
print(f"weights: {checkpoint_path(run_id)}")

In [ ]:
from pathlib import Path

from src.checkpoints import checkpoint_dir, checkpoint_path
from src.config import load_run_config, run_training
from src.experiments import log_run

SEED = 0
CONFIG = "configs/train/task3_hin_indicbert.yaml"

run = load_run_config(CONFIG, seed=SEED)
run_id = f"{Path(CONFIG).stem}_seed{SEED}"

metrics = run_training(run, device="cuda", work_dir=checkpoint_dir(run_id), progress=print)
log_run(run, run_id, metrics)

print(f"\nrun_id: {run_id}")
print(f"macro-F1 (test): {metrics['macro_f1']:.4f}  accuracy: {metrics['accuracy']:.4f}")
print(f"weights: {checkpoint_path(run_id)}")

### Where the fine-tuned weights land

Each run writes to `checkpoints/<run_id>/`, where `run_id` is
`<config-name>_seed<N>` (e.g. `task2_hin_indicbert_seed0`):

- **`checkpoint.pt`** — the full training state, rewritten after every epoch:
  model weights, optimiser/scheduler state, RNG state, and a snapshot of the
  **best** (highest dev macro-F1) epoch's weights under `blob["best"]["state"]`.
  `run_training` already reloads that best snapshot into the model before
  returning, so the metrics printed above are for the best epoch, not the last.
- **`run_config.json`** — the resolved config + its hash, for reference.

`checkpoints/` is **gitignored** — weights are large (~0.5 GB each) and
regenerable from a config + seed. `experiments.csv` and `reports/` are the
durable, committed record of what each run scored.

Retrieve any run later by its `run_id` alone (T-208):

In [ ]:
from src import checkpoints as C

for run_id in ("task2_hin_indicbert_seed0", "task3_hin_indicbert_seed0"):
    meta = C.metadata(run_id)
    print(run_id, "->", meta)

# weights = C.best_state_dict("task2_hin_indicbert_seed0")  # a state_dict to reload

### Compare every logged run

Reads back everything logged so far — from these cells or from
`scripts.t206_baseline`.

In [ ]:
import pandas as pd
from src.experiments import read_results

exp = pd.DataFrame(read_results())
display(exp)

try:
    baselines = pd.read_parquet("reports/baselines.parquet")
    summary = (baselines.groupby("config")["macro_f1"]
               .agg(["mean", "std", "count"]).round(4))
    display(summary)
except FileNotFoundError:
    print("no reports/baselines.parquet yet — run scripts.t206_baseline")

## T-207 — baseline gap

Load `experiments.csv`, compare the Hindi macro-F1 to IndicFinNLP's published
number, inspect misclassified rows to diagnose any gap before touching
hyperparameters.